# Trying to create and test U net 

# Arborescence de projet

In [1]:
import sys
import os
# On récupère le chemin absolu du dossier parent (..) et on l'ajoute au sys.path
dossier_parent = os.path.abspath('..')
if dossier_parent not in sys.path:
    sys.path.append(dossier_parent)

## Imports

In [2]:
import musdb
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from u_net import UnetAudioStemmer
from tool_box.pad_or_crop import pad_or_crop_to_multiple
from IPython.display import Audio, display

## Data

In [3]:
data_base = "../data/dataset/"
mus = musdb.DB(root=data_base,is_wav=False)
track = mus.tracks[0]
X = torch.from_numpy(track.audio.T)
y = torch.from_numpy(track.targets["vocals"].audio.T)
if X.shape[0] > 1: 
        X = torch.mean(X,dim=0,keepdim=True)
if y.shape[0] > 1: 
        y = torch.mean(y,dim=0,keepdim=True)
sample_rate = track.rate
# On ne prends que 3 secondes du morceau 
X = X[:,30 * sample_rate : 33 * sample_rate]
y = y[:,30 * sample_rate : 33 * sample_rate]
print(f"🎵 Morceau chargé : {track.name}")
print(f"⏱️ Durée : {track.duration} secondes")

display(Audio(X, rate=sample_rate))
display(Audio(y,rate=sample_rate))


🎵 Morceau chargé : A Classic Education - NightOwl
⏱️ Durée : 171.24 secondes


# Constants

In [4]:
n_fft = 2048
hop_length = 512

# STFT Conversion 

In [5]:
X_stft = torch.stft(X,n_fft=n_fft,hop_length=hop_length,return_complex=True)
y_stft = torch.stft(y,n_fft=n_fft,hop_length=hop_length,return_complex=True)

c:\Users\ggyor\miniconda3\envs\hf-audio-stem\lib\site-packages\torch\functional.py:704: UserWarning: A window was not provided. A rectangular window will be applied,which is known to cause spectral leakage. Other windows such as torch.hann_window or torch.hamming_window can are recommended to reduce spectral leakage.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\SpectralOps.cpp:842.)
  return _VF.stft(  # type: ignore[attr-defined]


# Passage en DB pour normaliser entre les différents instruments. 

In [6]:
X_db = 20 * torch.log10(torch.abs(X_stft) + 1e-5)
y_db = 20 * torch.log10(torch.abs(y_stft) + 1e-5)

# Correction pour normaliser sur les fréquences

In [7]:
X_db = X_db[:, :-1, :]
y_db= y_db[:, :-1, :]

# Ajout d'une dimension pour le batch

In [8]:
X_train = X_db.unsqueeze(0).float()
y_train = y_db.unsqueeze(0).float()
X_train = pad_or_crop_to_multiple(X_train,multiple=8)
y_train = pad_or_crop_to_multiple(y_train,multiple=8)

print(f"📊 Shape de l'entrée X (Mix) : {X_train.shape}")
print(f"📊 Shape de la cible Y (Voix) : {y_train.shape}")

📊 Shape de l'entrée X (Mix) : torch.Size([1, 1, 1024, 256])
📊 Shape de la cible Y (Voix) : torch.Size([1, 1, 1024, 256])


# Init du modèle

In [9]:
model = UnetAudioStemmer()

# Utilisation de la L1 loss et pas la MSE car la L1 permet d'éviter que le modèle se focus sur les grosses valeurs gros pics d'énergie dans la musique alors que nous on veut éliminer le bruit ambiant autour de la voix.
criterion = nn.L1Loss()

# Optimizer 
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Paramètres de l'entraînement 

In [10]:
epochs = 50 # On va répéter l'opération 50 fois sur le même extrait

# Entraînement

In [11]:
# Passer le modèle en mode train 
model.train()

for epoch in range(epochs):
    # Reset les gradients d'optimisation 
    optimizer.zero_grad()

    # Inference
    voice_spectogram = model(X_train)

    # Loss 
    loss = criterion(y_train, voice_spectogram)

    # Backward pass
    loss.backward()

    # Model Optimisation 
    optimizer.step()

    # Affichage de la progression tous les 10 epochs
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch + 1}/{epochs}] - Loss : {loss.item():.4f}")

print("✅ Entraînement terminé !")

voice_spectogram = model(X_train)
print("last spectrogram")
display(Audio(voice_spectogram, rate=sample_rate))


torch.Size([1, 1, 1024, 256])
torch.Size([1, 1, 512, 128])


RuntimeError: The size of tensor a (256) must match the size of tensor b (128) at non-singleton dimension 3